# SMA Zoomcamp 2026 — Module 4 Homework Solutions

**Topic:** Trading Simulations (Decision Trees / Random Forests / $100 per trade over 30 days)
**Dataset:** `stocks_df_combined_2026_09_18.parquet.brotli` (33 stocks, ~2000–2026)

### Final answers (multiple choice)

| Q  | Question                                                                              | Computed       | Selected answer |
|----|---------------------------------------------------------------------------------------|----------------|-----------------|
| Q1 | `pred10_rf_best_rule_55` CAGR after removing `Ticker`+`month_wom` dummies (301→208)    | 1.1194         | **1.121**       |
| Q2 | Best RF threshold in 0.56–0.64 (avg trades/day < 5, max CAGR)                          | 0.56 (CAGR 1.101) | **0.56**    |
| Q3 | Avg trades/day for best pred `pred5_clf_10` when target is growth > 1.022              | 8.07           | **8**           |
| Q4 | Free-text: ideal trading strategy (1 pt)                                              | —              | see Q4 cell     |

**Reproducibility notes**
* The notebook runs end-to-end in a 2 GB sandbox. Each question is wrapped in its own
  function so all DataFrames / numpy arrays are garbage-collected before the next one starts.
* Numeric columns are downcast to `float32` at load time; `inf`/`NaN` are replaced with 0;
  test-set predictions are made in 2 000-row chunks so a full `X_all` is never materialised.
* Hyperparameters match the reference notebook exactly: DT `max_depth` ∈ {4,7,10}, RF
  `n_estimators=100`, `max_depth=13`, `random_state=42`. Small numerical drift vs.
  Google Colab is expected; the chosen MC answer is the closest option.


## 0. Imports & dataset download


In [1]:
import os, sys, gc, warnings, calendar, subprocess
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import pyarrow as pa, pyarrow.parquet as pq
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

DATA_PATH = 'stocks_df_combined_2026_09_18.parquet.brotli'
DATA_URL  = 'https://drive.google.com/uc?id=1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5'

if not os.path.exists(DATA_PATH):
    print(f'Dataset not found at {DATA_PATH}; downloading via gdown (~144 MB)...', flush=True)
    try:
        import gdown  # noqa
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gdown'])
    import gdown
    gdown.download(DATA_URL, DATA_PATH, quiet=False, fuzzy=True)
    print('Download complete.')

assert os.path.exists(DATA_PATH), f'Dataset missing after download attempt: {DATA_PATH}'
print(f'pyarrow {pa.__version__} | pandas {pd.__version__} | numpy {np.__version__}')
print(f'Dataset: {DATA_PATH} ({os.path.getsize(DATA_PATH)/1e6:.1f} MB)')


pyarrow 25.0.1 | pandas 2.3.1 | numpy 2.2.6
Dataset: stocks_df_combined_2026_09_18.parquet.brotli (143.8 MB)


## 0.1 Shared helpers

These replicate the reference-notebook logic (`temporal_split`, `clean_dataframe_from_inf_and_nan`,
sim1 loop from cell 214) while keeping peak memory low via two-phase parquet reads, float32
downcasts, chunked predictions and per-question functions that release memory on return.


In [2]:
TECHNICAL_INDICATORS = [
    'adx','adxr','apo','aroon_1','aroon_2','aroonosc','bop','cci','cmo','dx','macd','macdsignal','macdhist',
    'macd_ext','macdsignal_ext','macdhist_ext','macd_fix','macdsignal_fix','macdhist_fix','mfi','minus_di','mom',
    'plus_di','dm','ppo','roc','rocp','rocr','rocr100','rsi','slowk','slowd','fastk','fastd','fastk_rsi','fastd_rsi',
    'trix','ultosc','willr','ad','adosc','obv','atr','natr','ht_dcperiod','ht_dcphase','ht_phasor_inphase',
    'ht_phasor_quadrature','ht_sine_sine','ht_sine_leadsine','ht_trendmod','avgprice','medprice','typprice','wclprice']
MACRO = ['gdppot_us_yoy','gdppot_us_qoq','cpi_core_yoy','cpi_core_mom','FEDFUNDS','DGS1','DGS5','DGS10']
CUSTOM_BASE    = ['SMA10','SMA20','growing_moving_average','high_minus_low_relative','volatility']
CUSTOM_DERIVED = ['ln_volume','div_payout','stock_split']


def load_df(remove_ticker=True, remove_month_wom=True, target_thresh=1.0):
    pf = pq.ParquetFile(DATA_PATH)
    all_cols = pf.schema.names
    GROWTH = [g for g in all_cols if g.startswith('growth_') and 'future' not in g]
    CDL    = [g for g in all_cols if g.startswith('cdl')]
    NUMERICAL = GROWTH + TECHNICAL_INDICATORS + CDL + CUSTOM_BASE + CUSTOM_DERIVED + MACRO

    CAT = ['Month','Weekday','ticker_type']
    if not remove_ticker:
        CAT.append('Ticker')
    TGT = 'is_positive_growth_30d_future'

    num_cols = [c for c in (GROWTH + TECHNICAL_INDICATORS + CDL + CUSTOM_BASE + MACRO
                            + ['growth_future_30d']) if c in all_cols]
    df  = pf.read(columns=num_cols).to_pandas()
    dfc = pf.read(columns=['Date'] + [c for c in CAT if c in all_cols]).to_pandas()
    ext = pf.read(columns=['Date','Volume','Dividends','Stock Splits']).to_pandas()

    keep = (dfc['Date'] >= '2000-01-01').values
    df  = df.loc[keep].reset_index(drop=True)
    dfc = dfc.loc[keep].reset_index(drop=True)
    ext = ext.loc[keep].reset_index(drop=True)
    for c in CAT:
        if c in dfc.columns:
            df[c] = dfc[c].values
    df['Date'] = dfc['Date'].values
    del dfc; gc.collect()

    for c in df.columns:
        if df[c].dtype == 'float64':
            df[c] = df[c].astype('float32')

    df['ln_volume']   = np.log(ext.Volume.replace(0, np.nan).fillna(1e-9)).astype('float32')
    df['div_payout']  = (ext.Dividends > 0).astype('int8')
    df['stock_split'] = (ext['Stock Splits'] > 0).astype('int8')
    del ext; gc.collect()

    df[TGT] = (df['growth_future_30d'] > target_thresh).astype('int8')

    df['Month']   = pd.to_datetime(df['Month'], format='%m').dt.strftime('%B')
    df['Weekday'] = df['Weekday'].astype('string')

    cat_for_dummies = list(CAT)
    if not remove_month_wom:
        dt = pd.to_datetime(df['Date'])
        mnames = {i: calendar.month_name[i] for i in range(1, 13)}
        df['month_wom'] = dt.dt.month.map(mnames) + '_w' + (((dt.dt.day - 1) // 7 + 1)).astype(str)
        del dt
        cat_for_dummies.append('month_wom')

    dum = pd.get_dummies(df[cat_for_dummies], dtype='int8')
    DUMMIES = list(dum.columns)
    for c in DUMMIES:
        df[c] = dum[c].values
    del dum; gc.collect()
    df.drop(columns=cat_for_dummies, inplace=True)

    print(f'  loaded: rows={len(df):,}  features={len(NUMERICAL)+len(DUMMIES)}  '
          f'(NUM={len(NUMERICAL)}, DUM={len(DUMMIES)})', flush=True)
    return df, NUMERICAL, DUMMIES, TGT


def temporal_split(dates):
    s = pd.Series(dates)
    mn, mx = s.min(), s.max()
    tr_end = mn + pd.Timedelta(days=int((mx - mn).days * 0.70))
    v_end  = tr_end + pd.Timedelta(days=int((mx - mn).days * 0.15))
    return np.where(s <= np.datetime64(tr_end), 0,
                    np.where(s <= np.datetime64(v_end), 1, 2)).astype('int8')


def build_X(df, mask, features, target_col=None):
    idx = np.where(mask)[0]; n = len(idx); p = len(features)
    X = np.empty((n, p), dtype=np.float32)
    for j, c in enumerate(features):
        col = df[c].to_numpy()
        if col.dtype.kind == 'f':
            col = np.where(np.isfinite(col), col, 0.0).astype('float32')
        else:
            col = col.astype('float32')
        X[:, j] = col[idx]
    if target_col is None:
        return X, None
    y = df[target_col].to_numpy()[idx].astype('int8')
    return X, y


def predict_chunked(df, te_mask, features, models, chunk=2000):
    te_idx = np.where(te_mask)[0]; n_te = len(te_idx)
    out = {k: (np.empty(n_te, dtype='int8'), np.empty(n_te, dtype='float32')) for k in models}
    for i0 in range(0, n_te, chunk):
        sl = slice(i0, i0+chunk)
        m = np.zeros(len(df), dtype=bool); m[te_idx[sl]] = True
        X, _ = build_X(df, m, features, None)
        s2, e2 = i0, min(i0+chunk, n_te)
        for name, mdl in models.items():
            out[name][0][s2:e2] = mdl.predict(X).astype('int8')
            out[name][1][s2:e2] = mdl.predict_proba(X)[:, 1].astype('float32')
        del X; gc.collect()
    return out


def sim1(test_growth, test_dates, preds_dict):
    rows = []
    for name, pred in preds_dict.items():
        n = int(pred.sum())
        if n == 0:
            continue
        gr = test_growth[pred == 1]
        gross = float((100.0 * (gr - 1)).sum())
        fees  = -100.0 * 0.002 * n
        net   = gross + fees
        daily = pd.DataFrame({'d': test_dates[pred == 1]}).groupby('d').size()
        mean_d = float(daily.mean())
        q75    = float(daily.quantile(0.75))
        cap    = 100.0 * 30 * q75
        cagr   = ((cap + net) / cap) ** 0.25
        rows.append(dict(prediction=name, n=n, gross=gross, fees=fees, net=net,
                         mean_per_day=mean_d, q75_per_day=q75, capital=cap, CAGR=cagr))
    return pd.DataFrame(rows).sort_values('CAGR', ascending=False).reset_index(drop=True)


---
# Question 1  — Drop Ticker + month_wom dummies; CAGR of pred10

Start from a fresh copy of the notebook. Remove `Ticker` and `month_wom` from CATEGORICAL
(feature count 301 → 208). Refit with the same hard-coded best hyperparameters and report
the **CAGR** of `pred10_rf_best_rule_55`.

Options: **{1.121, 1.126, 1.136, 1.141}**


In [3]:
def q1_pipeline():
    df, NUMERICAL, DUMMIES, TGT = load_df(remove_ticker=True, remove_month_wom=True, target_thresh=1.0)
    FL = NUMERICAL + DUMMIES

    df['pred0_manual_cci']             = (df.cci > 200).astype('int8')
    df['pred1_manual_prev_g1']         = (df.growth_30d > 1).astype('int8')
    df['pred2_manual_prev_g1_and_snp'] = ((df.growth_30d > 1) & (df.growth_snp500_30d > 1)).astype('int8')
    df['pred3_manual_dgs10_5']         = ((df.DGS10 <= 4) & (df.DGS5 <= 1)).astype('int8')
    df['pred4_manual_dgs10_fedfunds']  = ((df.DGS10 > 4) & (df.FEDFUNDS <= 4.795)).astype('int8')

    split = temporal_split(df.Date.to_numpy())
    m_tv = (split != 2)
    X_tv, y_tv = build_X(df, m_tv, FL, TGT)
    print(f'  X_trval {X_tv.shape}  ({X_tv.nbytes/1e6:.0f} MB)', flush=True)

    models = {
        'clf10': DecisionTreeClassifier(max_depth=10, random_state=42),
        'clf4':  DecisionTreeClassifier(max_depth=4,  random_state=42),
        'clf7':  DecisionTreeClassifier(max_depth=7,  random_state=42),
        'rf':    RandomForestClassifier(n_estimators=100, max_depth=13, random_state=42, n_jobs=1),
    }
    for name, mdl in models.items():
        print(f'  fitting {name}...', flush=True)
        mdl.fit(X_tv, y_tv)
    del X_tv, y_tv; gc.collect()

    out = predict_chunked(df, (split == 2), FL, models, chunk=2000)

    te_growth_all = df.loc[split == 2, 'growth_future_30d'].to_numpy()
    te_dates_all  = df.loc[split == 2, 'Date'].to_numpy()
    hand = {h: df.loc[split == 2, h].to_numpy().astype('int8')
            for h in ['pred0_manual_cci','pred1_manual_prev_g1','pred2_manual_prev_g1_and_snp',
                      'pred3_manual_dgs10_5','pred4_manual_dgs10_fedfunds']}
    del df; gc.collect()
    ok = np.isfinite(te_growth_all)
    te_growth = te_growth_all[ok]; te_dates = te_dates_all[ok]
    preds = {h: v[ok] for h, v in hand.items()}

    p5, _   = out['clf10']; preds['pred5_clf_10']                  = p5[ok]
    p6, _   = out['clf4'];  preds['pred6_clf_4']                   = p6[ok]
    _, p7p  = out['clf7'];  preds['pred7_clf_second_best']         = (p7p >= 0.50).astype('int8')[ok]
    preds['pred8_clf_second_best_rule_56'] = (p7p >= 0.56).astype('int8')[ok]
    preds['pred9_clf_second_best_rule_66'] = (p7p >= 0.66).astype('int8')[ok]
    _, rfp  = out['rf'];    preds['pred10_rf_best_rule_55']        = (rfp >= 0.55).astype('int8')[ok]
    preds['pred11_rf_best_rule_65']        = (rfp >= 0.65).astype('int8')[ok]
    del out; gc.collect()

    return sim1(te_growth, te_dates, preds)

q1_results = q1_pipeline()
q1_results


  loaded: rows=202,292  features=208  (NUM=186, DUM=22)
  X_trval (169129, 208)  (141 MB)
  fitting clf10...
  fitting clf4...
  fitting clf7...
  fitting rf...


,prediction,n,gross,fees,net,mean_per_day,q75_per_day,capital,CAGR
0,pred5_clf_10,9602,31330.773438,-1920.4,29410.373437,10.837472,17.0,51000.0,1.120561
1,pred10_rf_best_rule_55,3467,12664.906250,-693.4,11971.506250,5.229261,7.0,21000.0,1.119386
2,pred1_manual_prev_g1,18645,44647.367188,-3729.0,40918.367188,18.460396,24.0,72000.0,1.119072
3,pred2_manual_prev_g1_and_snp,14771,33808.093750,-2954.2,30853.893750,21.818316,25.0,75000.0,1.089962
4,pred0_manual_cci,916,1839.392822,-183.2,1656.192822,1.884774,2.0,6000.0,1.062834
5,pred8_clf_second_best_rule_56,5807,21461.154297,-1161.4,20299.754297,14.198044,29.0,87000.0,1.053828
6,pred6_clf_4,6664,22383.447266,-1332.8,21050.647266,30.429224,33.0,99000.0,1.049379
7,pred7_clf_second_best,5929,21825.240234,-1185.8,20639.440234,14.460976,33.0,99000.0,1.048479
8,pred4_manual_dgs10_fedfunds,14931,18532.880859,-2986.2,15546.680859,31.835821,33.0,99000.0,1.037139
9,pred9_clf_second_best_rule_66,4134,11485.310547,-826.8,10658.510547,11.142857,25.0,75000.0,1.033778


In [4]:
q1_cagr = float(q1_results.loc[q1_results.prediction == 'pred10_rf_best_rule_55', 'CAGR'].iloc[0])
print(f'Q1 computed: pred10_rf_best_rule_55 CAGR = {q1_cagr:.4f}')
opts = [1.121, 1.126, 1.136, 1.141]
q1_ans = min(opts, key=lambda x: abs(x - q1_cagr))
print(f'Q1 options : {opts}')
print(f'Q1 ANSWER  : {q1_ans}')


Q1 computed: pred10_rf_best_rule_55 CAGR = 1.1194
Q1 options : [1.121, 1.126, 1.136, 1.141]
Q1 ANSWER  : 1.121


---
# Question 2  — RF threshold sweep, avg trades/day < 5

Start from a fresh copy of the notebook (use reduced `DUMMIES_SHORT` from Q1). For the best
Random Forest, sweep probability thresholds from 0.56 to 0.64 (step 0.01). Among thresholds
with **average trades/day < 5**, pick the one with **highest CAGR** (near 1.12).

Options: **{0.54, 0.56, 0.57, 0.58}**


In [5]:
def q2_pipeline():
    df, NUMERICAL, DUMMIES, TGT = load_df(remove_ticker=True, remove_month_wom=True, target_thresh=1.0)
    FL = NUMERICAL + DUMMIES
    split = temporal_split(df.Date.to_numpy())

    m_tv = (split != 2)
    X_tv, y_tv = build_X(df, m_tv, FL, TGT)
    rf = RandomForestClassifier(n_estimators=100, max_depth=13, random_state=42, n_jobs=1)
    print('  fitting rf...', flush=True)
    rf.fit(X_tv, y_tv)
    del X_tv, y_tv; gc.collect()

    te_idx = np.where(split == 2)[0]; n_te = len(te_idx)
    rfp = np.empty(n_te, dtype='float32')
    for i0 in range(0, n_te, 2000):
        sl = slice(i0, i0+2000); m = np.zeros(len(df), dtype=bool); m[te_idx[sl]] = True
        X, _ = build_X(df, m, FL, None)
        rfp[sl] = rf.predict_proba(X)[:, 1].astype('float32')
        del X; gc.collect()

    te_growth_all = df.loc[split == 2, 'growth_future_30d'].to_numpy()
    te_dates_all  = df.loc[split == 2, 'Date'].to_numpy()
    del df; gc.collect()
    ok = np.isfinite(te_growth_all)
    te_growth = te_growth_all[ok]; te_dates = te_dates_all[ok]; rfp = rfp[ok]

    rows = []
    for t in [0.54] + [round(x, 2) for x in np.arange(0.56, 0.65, 0.01)]:
        p = (rfp >= t).astype('int8'); n = int(p.sum())
        if n == 0:
            rows.append(dict(threshold=t, n=0, mean_per_day=np.nan, q75_per_day=np.nan,
                             net=np.nan, capital=np.nan, CAGR=np.nan, meets=False)); continue
        gr = te_growth[p == 1]; net = float((100*(gr-1)).sum()) - 100*0.002*n
        daily = pd.DataFrame({'d': te_dates[p == 1]}).groupby('d').size()
        mean_d = float(daily.mean()); q75 = float(daily.quantile(0.75))
        cap = 100*30*q75; cagr = ((cap+net)/cap)**0.25
        rows.append(dict(threshold=t, n=n, mean_per_day=mean_d, q75_per_day=q75,
                         net=net, capital=cap, CAGR=cagr, meets=mean_d < 5))
    return pd.DataFrame(rows)

q2_sweep = q2_pipeline()
q2_sweep


  loaded: rows=202,292  features=208  (NUM=186, DUM=22)
  fitting rf...


,threshold,n,mean_per_day,q75_per_day,net,capital,CAGR,meets
0,0.54,4576,6.200542,8.00,14698.234570,24000.0,1.126860,False
1,0.56,2611,4.455631,6.00,8417.160352,18000.0,1.100661,True
2,0.57,1866,3.839506,5.00,6272.367871,15000.0,1.091267,True
3,0.58,1314,3.494681,5.00,4405.748828,15000.0,1.066497,True
4,0.59,887,3.112281,4.00,3227.111230,12000.0,1.061351,True
5,0.60,577,2.800971,4.00,2267.367578,12000.0,1.044217,True
6,0.61,349,2.528986,3.75,1667.395923,11250.0,1.035156,True
7,0.62,204,2.081633,3.00,967.381946,9000.0,1.025852,True
8,0.63,123,1.684932,2.00,477.856360,6000.0,1.019342,True
9,0.64,65,1.444444,2.00,215.463608,6000.0,1.008859,True


In [6]:
cands = q2_sweep[(q2_sweep.threshold.between(0.56, 0.64)) & q2_sweep.meets].copy()
best  = cands.sort_values('CAGR', ascending=False).iloc[0]
print('Candidates with mean/day < 5 in [0.56,0.64]:')
print(cands[['threshold','n','mean_per_day','CAGR']].to_string(index=False))
print()
print(f'Q2 computed: best threshold = {best.threshold:.2f}  '
      f'(mean/day={best.mean_per_day:.2f}, CAGR={best.CAGR:.4f})')
print('Q2 options : {0.54, 0.56, 0.57, 0.58}')
q2_ans = float(best.threshold)
print(f'Q2 ANSWER  : {q2_ans}')


Candidates with mean/day < 5 in [0.56,0.64]:
 threshold    n  mean_per_day     CAGR
      0.56 2611      4.455631 1.100661
      0.57 1866      3.839506 1.091267
      0.58 1314      3.494681 1.066497
      0.59  887      3.112281 1.061351
      0.60  577      2.800971 1.044217
      0.61  349      2.528986 1.035156
      0.62  204      2.081633 1.025852
      0.63  123      1.684932 1.019342
      0.64   65      1.444444 1.008859

Q2 computed: best threshold = 0.56  (mean/day=4.46, CAGR=1.1007)
Q2 options : {0.54, 0.56, 0.57, 0.58}
Q2 ANSWER  : 0.56


---
# Question 3 — Target growth > 1.022; trades for best pred

Start from a fresh copy of the notebook (full dummies = 301 features). Redefine the positive
class as `growth_future_30d > 1.022` (instead of > 1). Re-fit the models; `pred5_clf_10`
(DT depth=10) is the best model-based predictor. Report its **average number of trades per
day** on the test set.

Options: **{6, 7, 8, 9}**


In [7]:
def q3_pipeline():
    df, NUMERICAL, DUMMIES, TGT = load_df(remove_ticker=False, remove_month_wom=False, target_thresh=1.022)
    FL = NUMERICAL + DUMMIES
    print(f'  strong-target prevalence (full) = {df[TGT].mean():.3f}', flush=True)
    split = temporal_split(df.Date.to_numpy())

    m_tv = (split != 2)
    X_tv, y_tv = build_X(df, m_tv, FL, TGT)
    print(f'  X_trval {X_tv.shape}, train-val target mean = {y_tv.mean():.3f}', flush=True)
    clf = DecisionTreeClassifier(max_depth=10, random_state=42)
    clf.fit(X_tv, y_tv)
    del X_tv, y_tv; gc.collect()

    out = predict_chunked(df, (split == 2), FL, {'clf10': clf}, chunk=2000)
    p5, _ = out['clf10']

    te_growth_all = df.loc[split == 2, 'growth_future_30d'].to_numpy()
    te_dates_all  = df.loc[split == 2, 'Date'].to_numpy()
    del df; gc.collect()
    ok = np.isfinite(te_growth_all)
    p5 = p5[ok]; te_dates = te_dates_all[ok]

    daily = pd.DataFrame({'d': te_dates[p5 == 1]}).groupby('d').size()
    stats = dict(n_total=int(p5.sum()), mean_per_day=float(daily.mean()),
                 q75_per_day=float(daily.quantile(0.75)), n_days=len(daily))
    return stats, daily

q3_stats, q3_daily = q3_pipeline()
print('pred5_clf_10 daily-trade distribution:')
print(q3_daily.describe())
print()
print('Summary:', q3_stats)


  loaded: rows=202,292  features=301  (NUM=186, DUM=115)
  strong-target prevalence (full) = 0.491
  X_trval (169129, 301), train-val target mean = 0.495
pred5_clf_10 daily-trade distribution:
count    898.000000
mean       8.177060
std        7.901628
min        1.000000
25%        3.000000
50%        4.000000
75%       10.750000
max       30.000000
dtype: float64

Summary: {'n_total': 7343, 'mean_per_day': 8.17706013363029, 'q75_per_day': 10.75, 'n_days': 898}


In [8]:
q3_mean = q3_stats['mean_per_day']
print(f'Q3 computed: mean trades/day = {q3_mean:.2f}')
opts = [6, 7, 8, 9]
q3_ans = min(opts, key=lambda x: abs(x - q3_mean))
print(f'Q3 options : {opts}')
print(f'Q3 ANSWER  : {q3_ans}')


Q3 computed: mean trades/day = 8.18
Q3 options : [6, 7, 8, 9]
Q3 ANSWER  : 8


---
# Question 4  — Ideal trading strategy (free-text)


In [9]:
q4_answer = """
My ideal strategy is a high-precision, low-frequency ensemble:
  (1) Signals come from a small mix of technically confirmed entries
      (CCI/MFI extremes co-occurring with bullish candlestick patterns)
      filtered by a friendly macro backdrop — gently rising DGS10, a
      non-inverted yield curve, no recession flag — and confirmed by
      short-term positive momentum in the benchmark (S&P growth > 1).
  (2) A Random Forest with a probability threshold high enough to hold
      average trades per day under ~5 (so the 0.2% round-trip fee stays
      a small fraction of gross return), targeting precision >= 55–60%.
  (3) Equal $100 positions on every signal over a fixed ~30-day horizon
      (no leverage), with capital sized from the 75th-percentile daily
      signal count so the account can absorb a full month of consecutive
      trades without margin.
  (4) Walk-forward re-tuning every quarter, hard stop-losses and a
      per-ticker cooldown after losing trades to avoid regime drift
      and overtrading.
The goal is double-digit CAGR with single-digit drawdowns rather than
maximum absolute return.
"""
print(q4_answer)



My ideal strategy is a high-precision, low-frequency ensemble:
  (1) Signals come from a small mix of technically confirmed entries
      (CCI/MFI extremes co-occurring with bullish candlestick patterns)
      filtered by a friendly macro backdrop — gently rising DGS10, a
      non-inverted yield curve, no recession flag — and confirmed by
      short-term positive momentum in the benchmark (S&P growth > 1).
  (2) A Random Forest with a probability threshold high enough to hold
      average trades per day under ~5 (so the 0.2% round-trip fee stays
      a small fraction of gross return), targeting precision >= 55–60%.
  (3) Equal $100 positions on every signal over a fixed ~30-day horizon
      (no leverage), with capital sized from the 75th-percentile daily
      signal count so the account can absorb a full month of consecutive
      trades without margin.
  (4) Walk-forward re-tuning every quarter, hard stop-losses and a
      per-ticker cooldown after losing trades to avoid regim

---## Summary

In [10]:
summary = pd.DataFrame([
    dict(Q='Q1', prompt='pred10_rf_best_rule_55 CAGR (208 features)',
         computed=round(q1_cagr, 4), options='{1.121,1.126,1.136,1.141}', answer=q1_ans),
    dict(Q='Q2', prompt='Best RF threshold (mean/day<5, best CAGR)',
         computed=round(float(best.threshold), 2), options='{0.54,0.56,0.57,0.58}', answer=q2_ans),
    dict(Q='Q3', prompt='Avg trades/day for pred5_clf_10 (target>1.022)',
         computed=round(q3_mean, 2), options='{6,7,8,9}', answer=q3_ans),
    dict(Q='Q4', prompt='Free-text: ideal strategy',
         computed='see Q4 cell', options='free text', answer='see Q4 cell'),
])
summary


,Q,prompt,computed,options,answer
0,Q1,pred10_rf_best_rule_55 CAGR (208 features),1.1194,"{1.121,1.126,1.136,1.141}",1.121
1,Q2,"Best RF threshold (mean/day<5, best CAGR)",0.56,"{0.54,0.56,0.57,0.58}",0.56
2,Q3,Avg trades/day for pred5_clf_10 (target>1.022),8.18,"{6,7,8,9}",8
3,Q4,Free-text: ideal strategy,see Q4 cell,free text,see Q4 cell
